# Capítulo 2 · Estados de Bell y Entrelazamiento Cuántico

## Objetivos

1. Construir los cuatro estados de Bell como estados máximamente entrelazados de dos qubits.
2. Verificar que ningún estado de Bell puede factorizarse como producto tensorial.
3. Calcular la entropía de Von Neumann de los estados reducidos como medida de entrelazamiento.
4. Simular el protocolo de teleportación cuántica.

---

## 2.1 Estados de Bell

Los cuatro estados de Bell forman una base ortonormal del espacio $\mathbb{C}^2 \otimes \mathbb{C}^2$:

$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}, \quad |\Phi^-\rangle = \frac{|00\rangle - |11\rangle}{\sqrt{2}}$$

$$|\Psi^+\rangle = \frac{|01\rangle + |10\rangle}{\sqrt{2}}, \quad |\Psi^-\rangle = \frac{|01\rangle - |10\rangle}{\sqrt{2}}$$

El estado $|\Phi^+\rangle$ se obtiene aplicando la puerta de Hadamard al primer qubit y después una CNOT:

$$|\Phi^+\rangle = \text{CNOT}_{01} \cdot (H \otimes I) \cdot |00\rangle$$

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from src.quantum_math import QuantumMath
from src.quantum_gates import Gates
from src.visualization import QuantumVisualization

print('Módulos cargados.')

In [ ]:
# ── Construcción manual de los estados de Bell ────────────────────

# Base computacional de 2 qubits
ket00 = QuantumMath.tensor_product(QuantumMath.ket0(), QuantumMath.ket0())
ket01 = QuantumMath.tensor_product(QuantumMath.ket0(), QuantumMath.ket1())
ket10 = QuantumMath.tensor_product(QuantumMath.ket1(), QuantumMath.ket0())
ket11 = QuantumMath.tensor_product(QuantumMath.ket1(), QuantumMath.ket1())

# Los cuatro estados de Bell
Phi_plus  = (ket00 + ket11) / np.sqrt(2)
Phi_minus = (ket00 - ket11) / np.sqrt(2)
Psi_plus  = (ket01 + ket10) / np.sqrt(2)
Psi_minus = (ket01 - ket10) / np.sqrt(2)

bell_states = {
    '|Φ+〉': Phi_plus,
    '|Φ-〉': Phi_minus,
    '|Ψ+〉': Psi_plus,
    '|Ψ-〉': Psi_minus,
}

for name, state in bell_states.items():
    print(f'{name} = {np.round(state, 4)}')

## 2.2 Generación con Qiskit

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

def bell_circuit(variant: str = 'Phi+') -> QuantumCircuit:
    """Construye el circuito que prepara uno de los cuatro estados de Bell."""
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    if variant in ('Phi-', 'Psi-'):
        qc.z(0)
    if variant in ('Psi+', 'Psi-'):
        qc.x(0)
    qc.cx(0, 1)
    return qc

# Preparar |Φ+〉
qc_bell = bell_circuit('Phi+')
print(qc_bell.draw('text'))

# Vector de estado analítico
sv = Statevector(qc_bell)
print('\nVector de estado |Φ+〉:')
print(np.round(sv.data, 4))

In [ ]:
# Medición del estado de Bell y visualización
qc_meas = bell_circuit('Phi+')
qc_meas.measure([0, 1], [0, 1])

backend = AerSimulator()
job = backend.run(qc_meas, shots=4096)
counts = job.result().get_counts()
print('Conteos:', counts)

fig = QuantumVisualization.plot_histogram(
    counts, title='Medidas del estado de Bell |Φ+〉 (4096 shots)'
)
plt.show()

## 2.3 Verificación de no-separabilidad

Un estado $|\psi\rangle \in \mathcal{H}_A \otimes \mathcal{H}_B$ es separable si y sólo si existe la factorización $|\psi\rangle = |a\rangle \otimes |b\rangle$. Esto equivale a que el **rango de Schmidt** sea 1, o equivalentemente, a que la **entropía de entrelazamiento** sea $S = 0$.

La entropía de entrelazamiento se calcula trazando sobre uno de los subsistemas:

$$S(A) = -\mathrm{Tr}(\rho_A \log_2 \rho_A)$$

Para $|\Phi^+\rangle$, se obtiene $S = 1$ bit, el máximo posible para dos qubits.

In [ ]:
def entanglement_entropy(state: np.ndarray, dim_A: int = 2) -> float:
    """Calcula la entropía de entrelazamiento S(A) para un estado bipartito.
    
    Parámetros
    ----------
    state : np.ndarray
        Vector de estado del sistema compuesto (2^n,).
    dim_A : int
        Dimensión del subsistema A.
    """
    dim_total = len(state)
    dim_B = dim_total // dim_A
    # Reformatear como matriz y aplicar SVD
    M = state.reshape(dim_A, dim_B)
    singular_values = np.linalg.svd(M, compute_uv=False)
    lambdas = singular_values ** 2   # autovalores de rho_A
    lambdas = lambdas[lambdas > 1e-14]
    return float(-np.sum(lambdas * np.log2(lambdas)))

print('Entropías de entrelazamiento:')
for name, state in bell_states.items():
    S = entanglement_entropy(state)
    print(f'  S({name}) = {S:.4f} bits')

# Estado producto (separable) para comparación
product_state = QuantumMath.tensor_product(
    QuantumMath.ket_plus(), QuantumMath.ket0()
)
S_prod = entanglement_entropy(product_state)
print(f'  S(|+〉⊗|0〉) = {S_prod:.4f} bits  ← estado producto, sin entrelazamiento')

## 2.4 Teleportación cuántica

El protocolo de teleportación permite transmitir el estado desconocido $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ de Alice a Bob usando un par de Bell compartido y dos bits clásicos de comunicación.

In [ ]:
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit.quantum_info import Statevector

def teleportation_circuit(alpha: complex, beta: complex) -> QuantumCircuit:
    """Circuito de teleportación cuántica.
    
    Qubits:
      q[0] = estado de Alice |ψ〉
      q[1] = qubit de Alice del par de Bell
      q[2] = qubit de Bob del par de Bell
    """
    q = QuantumRegister(3, 'q')
    c0 = ClassicalRegister(1, 'c0')
    c1 = ClassicalRegister(1, 'c1')
    qc = QuantumCircuit(q, c0, c1)

    # Paso 0: preparar |ψ〉 en q[0]
    qc.initialize([alpha, beta], 0)
    qc.barrier(label='Preparación')

    # Paso 1: crear par de Bell entre q[1] y q[2]
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier(label='Par de Bell')

    # Paso 2: operaciones de Alice
    qc.cx(0, 1)
    qc.h(0)
    qc.barrier(label='Alice')

    # Paso 3: medida de Alice
    qc.measure(0, c0)
    qc.measure(1, c1)
    qc.barrier(label='Medida')

    # Paso 4: correcciones de Bob (con if clásico)
    with qc.if_test((c1, 1)):
        qc.x(2)
    with qc.if_test((c0, 1)):
        qc.z(2)

    return qc

# Ejecutar con un estado de prueba
import numpy as np
theta_test = np.radians(70)
a = np.cos(theta_test / 2)             # alpha real
b = np.exp(1j * np.radians(30)) * np.sin(theta_test / 2)  # beta con fase

qc_tel = teleportation_circuit(a, b)
print('Circuito de teleportación:')
print(qc_tel.draw('text'))

In [ ]:
# Verificar que el qubit de Bob recibe el estado correcto
# (simulación statevector antes de la medida)
qc_no_meas = QuantumCircuit(3)
qc_no_meas.initialize([a, b], 0)
qc_no_meas.h(1)
qc_no_meas.cx(1, 2)
qc_no_meas.cx(0, 1)
qc_no_meas.h(0)

sv_pre = Statevector(qc_no_meas)
print(f'Estado original |ψ〉: α={a:.4f}, β={b:.4f}')
print(f'Fidelidad esperada en teleportación: 1.0 (protocolo determinista clásico)')
print('\n→ El protocolo garantiza que Bob recupera el estado exacto de Alice')
print('  a costa de destruirlo en el lado de Alice (no-clonación).')

## 2.5 Ejercicios propuestos

1. Verifica que los cuatro estados de Bell son ortonormales calculando todos los productos internos $\langle \Phi^+ | \Phi^- \rangle$, $\langle \Phi^+ | \Psi^+ \rangle$, etc.

2. Construye el circuito que genera $|\Psi^-\rangle$ y mide 4096 shots. ¿Cuántos '00' y '11' observas? ¿Por qué?

3. ¿Puede entrelazarse un estado de tres qubits $|\psi\rangle \in (\mathbb{C}^2)^{\otimes 3}$ de tal forma que la traza sobre cualquier subconjunto de qubits sea un estado máximamente mezclado? Investiga los estados GHZ y W.

4. Modifica el circuito de teleportación para teleportar el estado $|-\rangle$. Verifica que Bob recupera el estado correcto.